# 🌳 Decision Trees with Scikit-Learn
### A Hands-On Worksheet — *Estimated time: 20–30 minutes*

---

In this worksheet you will:
1. Load **real-world datasets** built into scikit-learn
2. Apply a **Decision Tree Regressor** (predicting a continuous value)
3. Apply a **Decision Tree Classifier** (predicting a category)
4. Evaluate each model and tune a key hyperparameter

> 📌 **How to use this sheet:** Every cell marked `# ✏️ TODO` requires you to write or complete code. Hints are provided in the cells just above. Run each cell in order (Shift+Enter).

---
## 📦 Step 0 — Import Libraries

We need:
- `pandas` / `numpy` — data wrangling
- `sklearn` — machine learning toolkit
- `matplotlib` — visualisation

Run the cell below. You don't need to change anything here.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (10, 4)

# Datasets
from sklearn.datasets import load_diabetes, load_iris

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Decision Trees
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree

# Metrics
from sklearn.metrics import (
    mean_squared_error, r2_score,          # regression
    accuracy_score, classification_report,  # classification
    ConfusionMatrixDisplay
)

print('✅ All libraries imported successfully!')

---
# 🔵 PART 1 — Decision Tree Regression
## Dataset: Diabetes (442 patients)

**Goal:** Predict a *quantitative* measure of diabetes disease progression one year after baseline.

**Features (10 input variables):**  
age, sex, BMI, average blood pressure, and six blood serum measurements.

**Target:** A continuous numeric score — so this is a **regression** problem.

---

### Step 1.1 — Load & Explore the Data

🔍 **What to look for:**
- Shape: how many rows (patients) and columns (features)?
- Any missing values?
- What does the target distribution look like?

In [ ]:
# Load the dataset — scikit-learn returns a Bunch object (like a dict)
diabetes = load_diabetes()

# Convert to a DataFrame for easy inspection
X_reg = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
y_reg = pd.Series(diabetes.target, name='disease_progression')

print('Dataset shape:', X_reg.shape)
print('\nFirst 5 rows:')
display(X_reg.head())

print('\nTarget variable — first 5 values:')
print(y_reg.head())

print('\nMissing values per column:')
print(X_reg.isnull().sum())

In [ ]:
# Visualise the target distribution
plt.hist(y_reg, bins=30, color='steelblue', edgecolor='white')
plt.title('Distribution of Disease Progression Score (target)')
plt.xlabel('Score')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

print(f'Target range: {y_reg.min():.0f} → {y_reg.max():.0f}  |  Mean: {y_reg.mean():.1f}')

### Step 1.2 — Split the Data

💡 **Why split?**  
We train the model on one portion and **test** it on unseen data — otherwise we'd be measuring memorisation, not learning.

A common split is **80% train / 20% test**.

---
**`train_test_split` key arguments:**
| Argument | Meaning |
|---|---|
| `test_size` | Fraction of data for testing (e.g. `0.2` = 20%) |
| `random_state` | Seed for reproducibility — any integer |


In [ ]:
# ✏️ TODO: Split X_reg and y_reg into training and test sets
# Use test_size=0.2 and random_state=42

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    ___,         # features
    ___,         # target
    test_size=___,
    random_state=___
)

print(f'Training samples : {len(X_train_r)}')
print(f'Test samples     : {len(X_test_r)}')

### Step 1.3 — Train the Decision Tree Regressor

💡 **Key hyperparameter — `max_depth`**  
Controls how deep the tree can grow.
- Too **shallow** → underfitting (misses patterns)
- Too **deep** → overfitting (memorises training data)

Start with `max_depth=3` for a simple, interpretable tree.

In [ ]:
# ✏️ TODO: Create a DecisionTreeRegressor with max_depth=3 and random_state=42
# Then fit (train) it on the training data

reg_tree = DecisionTreeRegressor(max_depth=___, random_state=___)
reg_tree.fit(___, ___)   # fit(X_train, y_train)

print('✅ Regressor trained!')

### Step 1.4 — Evaluate the Regressor

**Two standard regression metrics:**

| Metric | Formula | Interpretation |
|---|---|---|
| **RMSE** (Root Mean Squared Error) | √(mean of squared errors) | Same unit as target; lower = better |
| **R²** (R-squared) | 1 − SS_res/SS_tot | 1.0 = perfect; 0 = predicts mean; <0 = worse than mean |


In [ ]:
# ✏️ TODO: Use the trained model to predict on the TEST set, then compute RMSE and R²

y_pred_r = reg_tree.predict(___)   # predict on X_test_r

rmse = np.sqrt(mean_squared_error(___, ___))   # (y_true, y_pred)
r2   = r2_score(___, ___)                      # (y_true, y_pred)

print(f'RMSE : {rmse:.2f}')
print(f'R²   : {r2:.3f}')

In [ ]:
# Visualise: Actual vs Predicted
plt.scatter(y_test_r, y_pred_r, alpha=0.6, color='steelblue', edgecolor='white', s=50)
lims = [y_test_r.min(), y_test_r.max()]
plt.plot(lims, lims, 'r--', label='Perfect prediction')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Regression: Actual vs Predicted')
plt.legend()
plt.tight_layout()
plt.show()

### Step 1.5 — Visualise the Tree

One of the biggest advantages of decision trees is **interpretability** — you can literally read the rules it learned!

Each node shows:
- The **split condition** (e.g. `bmi <= 0.009`)
- **mse** — impurity at that node
- **samples** — how many training points reached this node
- **value** — the predicted value if we stopped here

In [ ]:
fig, ax = plt.subplots(figsize=(18, 7))
plot_tree(
    reg_tree,
    feature_names=diabetes.feature_names,
    filled=True,
    rounded=True,
    fontsize=9,
    ax=ax
)
plt.title('Decision Tree Regressor (max_depth=3)', fontsize=14)
plt.tight_layout()
plt.show()

### 🔬 Step 1.6 — Depth Experiment (Overfitting vs Underfitting)

Let's see how `max_depth` affects train vs test performance.

🧠 **What to observe:**
- Training R² always increases with depth
- Test R² peaks at some optimal depth, then **drops** (= overfitting)
- The gap between train and test R² is called the **generalisation gap**

In [ ]:
depths = range(1, 15)
train_r2, test_r2 = [], []

for d in depths:
    m = DecisionTreeRegressor(max_depth=d, random_state=42)
    m.fit(X_train_r, y_train_r)
    train_r2.append(r2_score(y_train_r, m.predict(X_train_r)))
    test_r2.append(r2_score(y_test_r,  m.predict(X_test_r)))

plt.plot(depths, train_r2, 'o-', label='Train R²', color='steelblue')
plt.plot(depths, test_r2,  's-', label='Test R²',  color='tomato')
plt.xlabel('max_depth')
plt.ylabel('R² Score')
plt.title('Overfitting Curve — Regression')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

best_depth = depths[np.argmax(test_r2)]
print(f'🏆 Best max_depth for test R²: {best_depth}  (R² = {max(test_r2):.3f})')

### ✍️ Reflection 1
Answer these questions in the cell below (as a comment or markdown):

1. At what `max_depth` did your model start overfitting? How can you tell?
2. What was the best test R² you achieved? Is this a good score for medical data?
3. Which feature appeared at the **root** (top) of the tree? Why do you think that is?

*✏️ Write your answers here...*

1. 
2. 
3. 

---
# 🟠 PART 2 — Decision Tree Classification
## Dataset: Iris Flowers (150 samples)

**Goal:** Classify iris flowers into **3 species** based on measurements.

**Features (4 input variables):**
- `sepal length (cm)`, `sepal width (cm)`
- `petal length (cm)`, `petal width (cm)`

**Target:** Species — `Setosa (0)`, `Versicolor (1)`, `Virginica (2)` → **classification** problem.

---

### Step 2.1 — Load & Explore the Data

In [ ]:
iris = load_iris()

X_clf = pd.DataFrame(iris.data, columns=iris.feature_names)
y_clf = pd.Series(iris.target, name='species')

print('Dataset shape:', X_clf.shape)
print('\nClass names  :', iris.target_names)
print('\nClass distribution:')
print(y_clf.value_counts().rename(dict(enumerate(iris.target_names))))

display(X_clf.describe().round(2))

In [ ]:
# Quick scatter of the two most informative features
colors = ['#E74C3C', '#2ECC71', '#3498DB']
for cls in range(3):
    mask = y_clf == cls
    plt.scatter(
        X_clf.loc[mask, 'petal length (cm)'],
        X_clf.loc[mask, 'petal width (cm)'],
        label=iris.target_names[cls],
        color=colors[cls], alpha=0.7, edgecolor='white', s=60
    )
plt.xlabel('Petal Length (cm)')
plt.ylabel('Petal Width (cm)')
plt.title('Iris Classes by Petal Dimensions')
plt.legend()
plt.tight_layout()
plt.show()

### Step 2.2 — Split the Data

💡 For classification, pass `stratify=y` to `train_test_split`.  
This ensures **each class has the same proportion** in both train and test sets — important when classes are imbalanced.

In [ ]:
# ✏️ TODO: Split X_clf and y_clf — use test_size=0.2, random_state=42, AND stratify=y_clf

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    ___,
    ___,
    test_size=___,
    random_state=___,
    stratify=___      # keeps class balance
)

print(f'Train: {len(X_train_c)} samples | Test: {len(X_test_c)} samples')
print('\nClass balance in test set:')
print(pd.Series(y_test_c).value_counts())

### Step 2.3 — Train the Decision Tree Classifier

💡 **`criterion` parameter** controls how the tree measures impurity at each split:
- `'gini'` — Gini Impurity (default) — fast, works well in practice
- `'entropy'` — Information Gain — can produce slightly different trees

Both usually give similar results. We'll use `'gini'`.

In [ ]:
# ✏️ TODO: Create a DecisionTreeClassifier with max_depth=3, criterion='gini', random_state=42
# Then fit it on the training data

clf_tree = DecisionTreeClassifier(
    max_depth=___,
    criterion=___,
    random_state=___
)
clf_tree.fit(___, ___)

print('✅ Classifier trained!')

### Step 2.4 — Evaluate the Classifier

**Key classification metrics:**

| Metric | What it measures |
|---|---|
| **Accuracy** | % of predictions that were correct |
| **Precision** | Of all predicted positives, how many were truly positive? |
| **Recall** | Of all actual positives, how many did we catch? |
| **F1-Score** | Harmonic mean of precision & recall |

A **Confusion Matrix** shows exactly which classes the model confuses.

In [ ]:
# ✏️ TODO: Predict on the test set and print accuracy and classification report

y_pred_c = clf_tree.predict(___)   # predict on X_test_c

acc = accuracy_score(___, ___)     # (y_true, y_pred)

print(f'Accuracy: {acc:.2%}\n')
print('Classification Report:')
print(classification_report(___, ___, target_names=iris.target_names))

In [ ]:
# Plot the Confusion Matrix
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test_c,
    y_pred_c,
    display_labels=iris.target_names,
    cmap='Blues',
    ax=ax
)
plt.title('Confusion Matrix — Iris Classifier')
plt.tight_layout()
plt.show()

### Step 2.5 — Visualise the Classification Tree

Each leaf node is colour-coded by class. The **darker** the shade, the **purer** the node (all samples belong to one class).

In [ ]:
fig, ax = plt.subplots(figsize=(18, 7))
plot_tree(
    clf_tree,
    feature_names=iris.feature_names,
    class_names=iris.target_names,
    filled=True,
    rounded=True,
    fontsize=10,
    ax=ax
)
plt.title('Decision Tree Classifier (max_depth=3)', fontsize=14)
plt.tight_layout()
plt.show()

### Step 2.6 — Feature Importance

💡 After training, `.feature_importances_` tells you how much each feature contributed to reducing impurity across all splits.  
Values sum to 1. Higher = more important.

In [ ]:
# ✏️ TODO: Extract feature importances and create a bar chart
# Hint: clf_tree.feature_importances_ returns an array aligned with iris.feature_names

importances = clf_tree.feature_importances_
feat_names  = iris.feature_names

# Sort by importance (descending)
sorted_idx = np.argsort(importances)[::-1]

plt.bar(
    [feat_names[i] for i in sorted_idx],
    importances[sorted_idx],
    color=['#3498DB', '#2ECC71', '#E74C3C', '#F39C12']
)
plt.title('Feature Importances — Iris Classifier')
plt.ylabel('Importance Score')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

for i in sorted_idx:
    print(f'  {feat_names[i]:<30} {importances[i]:.4f}')

### 🔬 Step 2.7 — Depth Experiment for Classification

In [ ]:
# ✏️ TODO: Loop over max_depth values 1–10 and record train & test accuracy
# Hint: structure is the same as the regression experiment in Step 1.6

depths = range(1, 11)
train_acc, test_acc = [], []

for d in depths:
    m = DecisionTreeClassifier(max_depth=___, random_state=42)
    m.fit(___, ___)
    train_acc.append(accuracy_score(___, m.predict(___)  ))
    test_acc.append( accuracy_score(___, m.predict(___)  ))

plt.plot(depths, train_acc, 'o-', label='Train Accuracy', color='steelblue')
plt.plot(depths, test_acc,  's-', label='Test Accuracy',  color='tomato')
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.title('Overfitting Curve — Classification')
plt.ylim(0, 1.05)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### ✍️ Reflection 2

1. Which two species did the classifier confuse most? Does the scatter plot from Step 2.1 help explain why?
2. Which feature had the highest importance? Does the tree diagram confirm this?
3. At what depth did overfitting start for the classifier?

*✏️ Write your answers here...*

1. 
2. 
3. 

---
# 🌟 BONUS Challenges

Finished early? Try one or more of these:

1. **Change the criterion** — Re-train the classifier with `criterion='entropy'`. Does accuracy change?

2. **Add `min_samples_leaf`** — This prevents a node from splitting if it would create a leaf with fewer than N samples. Try `min_samples_leaf=5`. How does it affect overfitting?

3. **Predict a new sample** — Create a NumPy array with custom petal/sepal measurements and predict its iris species. Use `clf_tree.predict([[...]])` and `clf_tree.predict_proba([[...]])` to see class probabilities.

4. **Try a different dataset** — Replace the Iris dataset with `load_wine()` (3-class wine classification). Does the same `max_depth=3` tree still perform well?

In [ ]:
# 🌟 Bonus workspace — write your experiments here


---
## 📋 Summary Cheat-Sheet

| | Regression | Classification |
|---|---|---|
| **Class** | `DecisionTreeRegressor` | `DecisionTreeClassifier` |
| **Loss/criterion** | `mse` (default) | `gini` or `entropy` |
| **Key metrics** | RMSE, R² | Accuracy, F1, Confusion Matrix |
| **Overfitting check** | Train R² ↑ while Test R² ↓ | Train Acc ↑ while Test Acc ↓ |
| **Main hyperparameter** | `max_depth` | `max_depth` |

**Workflow (always the same):**
```
Load data → Split → Instantiate model → .fit() → .predict() → Evaluate → Tune
```

> 🚀 Next steps: **Random Forests** (many trees voting together) dramatically reduce overfitting — try `sklearn.ensemble.RandomForestClassifier`!